# EXP-5 — FlashRank Reranking

Fetch a larger candidate set (top-10) using vector search, then rerank using FlashRank to keep only the top-3. FlashRank is a lightweight local model, no extra API calls needed.

## Setup

In [1]:
import os
import sys
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

# using the new-era langchain packages, not the old monolith
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings("ignore")
print("all imports loaded")


c:\projects\Rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\projects\Rag-project\.venv\Lib\site-packages\ragas\metrics\__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from ragas.metrics._answer_correctness import AnswerCorrectness, answer_correctness
c:\projects\Rag-project\.venv\Lib\site-

all imports loaded


In [2]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [3]:
# langsmith traces every chain.invoke() automatically once these env vars are set
# you will see each run in your langsmith project dashboard in real time
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [4]:
# set MLFLOW_TRACKING_URI in your .env to point to a remote mlflow server
# if running locally first: mlflow server --host 0.0.0.0 --port 5000
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [5]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [7]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1313.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [8]:
print(f"eval set: {len(dataset['question'])} question-answer pairs")

eval set: 10 question-answer pairs


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [11]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [13]:
import nest_asyncio
nest_asyncio.apply()

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            docs = get_docs_fn(question)
            answers.append(answer)
            contexts.append([d.page_content for d in docs])
        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    })

    # max_workers=1 means sequential — no parallel calls, no rate limit hits
    # timeout=120 because Groq free tier can be slow sometimes
    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,   # if one question fails,we still want to get metrics on the rest of the eval set
    )
    return result

In [14]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)

In [21]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment-Specific Imports

In [16]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

## Experiment

In [17]:
base_retriever = db.as_retriever(search_kwargs={"k": 10})

# flashrank runs locally, much faster than cross-encoder but slightly less accurate
compressor = FlashrankRerank(top_n=3)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

chain = (
    {"context": rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The leave policy is outlined in the Leave Management Policy document (HRP-001) of TechCorp India Pvt. Ltd. It includes various types of leaves such as Earned Leave / Privilege Leave (PL/EL) and Casual Leave (CL), with specific entitlements, accrual rates, carry-forward rules, and encashment options.


In [18]:
result = evaluate_rag(chain, rerank_retriever.invoke, dataset)
latency = measure_latency(chain)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 0.398206 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.gro

In [19]:
result,latency

({'faithfulness': 0.6602, 'context_precision': 0.7333, 'context_recall': 0.4000},
 9.307)

In [22]:
log_to_mlflow(
    run_name="EXP-5-FLASHRANK",
    result=result,
    latency=latency,
    retriever_type="flashrank-rerank",
    top_k=3,
    extra_params={"initial_k": 10, "rerank_top_n": 3},
)


🏃 View run EXP-5-FLASHRANK at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/1c49c78f02bc47239da3e9ecdf021f97
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
